La CNN-1D es una buena candidata para cerrar la comparativa de modelos de deep learning del TFG.

Frente al MLP, puede capturar patrones locales entre variables consecutivas dentro de una ventana. Frente a la LSTM, suele ser mas ligera y mas facil de entrenar. En este notebook seguimos el mismo patron general: Optuna multiobjetivo con F1 y latencia, seleccion de candidatos desde la frontera de Pareto, evaluacion final en test y benchmark de recursos del modelo ganador.

In [ ]:
import os
import time
import numpy as np
import polars as pl
import optuna
import psutil
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow import keras
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, roc_curve, auc
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler

HAS_GPU = len(tf.config.list_physical_devices('GPU')) > 0
TRAIN_DEVICE = '/GPU:0' if HAS_GPU else '/CPU:0'
INFER_DEVICE = '/CPU:0'

if HAS_GPU:
    print('GPU detectada. El entrenamiento se ejecutara en GPU y la inferencia se medira en CPU.')
else:
    print('No hay GPU disponible. Entrenamiento e inferencia se ejecutaran en CPU.')

tf.keras.backend.clear_session()


In [ ]:
# ==========================================
# 1. FUNCIONES AUXILIARES Y CARGA DE DATOS
# ==========================================

def create_sequences(X, y, time_steps):
    Xs, ys = [], []
    for i in range(len(X) - time_steps + 1):
        Xs.append(X[i:(i + time_steps)])
        ys.append(y[i + time_steps - 1])
    return np.array(Xs), np.array(ys)


DEFAULT_DROPOUT_RATE = 0.2


def build_cnn1d_model(time_steps, n_features, n_filters, kernel_size, dense_units, dropout_rate=DEFAULT_DROPOUT_RATE):
    model = keras.Sequential([
        keras.layers.Input(shape=(time_steps, n_features)),
        keras.layers.Conv1D(filters=n_filters, kernel_size=kernel_size, padding='same', activation='relu'),
        keras.layers.MaxPooling1D(pool_size=2),
        keras.layers.Conv1D(filters=max(16, n_filters // 2), kernel_size=kernel_size, padding='same', activation='relu'),
        keras.layers.GlobalMaxPooling1D(),
        keras.layers.Dense(dense_units, activation='relu'),
        keras.layers.Dropout(dropout_rate),
        keras.layers.Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


def clone_model_to_cpu(trained_model, time_steps, n_features, n_filters, kernel_size, dense_units, dropout_rate):
    with tf.device(INFER_DEVICE):
        cpu_model = build_cnn1d_model(
            time_steps=time_steps,
            n_features=n_features,
            n_filters=n_filters,
            kernel_size=kernel_size,
            dense_units=dense_units,
            dropout_rate=dropout_rate
        )
        cpu_model.set_weights(trained_model.get_weights())
    return cpu_model


path_train = '../DATASETS/dataSets_Reducidos/nusw-nb15/datos_train_NUSW_redux.csv'
path_test  = '../DATASETS/dataSets_Reducidos/nusw-nb15/datos_test_NUSW_redux.csv'

df_train = pl.read_csv(path_train)
df_test  = pl.read_csv(path_test)
TARGET_COL = 'attack_cat'

y_train = (
    df_train.select(
        pl.when(pl.col(TARGET_COL).str.strip_chars() == 'Normal')
        .then(1)
        .otherwise(-1)
        .alias('label')
    )
    .to_series()
    .cast(pl.Int8)
)

y_test = (
    df_test.select(
        pl.when(pl.col(TARGET_COL).str.strip_chars() == 'Normal')
        .then(1)
        .otherwise(-1)
        .alias('label')
    )
    .to_series()
    .cast(pl.Int8)
)

x_train = df_train.drop(TARGET_COL)
x_test = df_test.drop(TARGET_COL)

X_full_train = x_train.to_numpy()
y_full_train = y_train.to_numpy()
X_test_np = x_test.to_numpy()
y_test_np = y_test.to_numpy()

y_full_train_01 = ((y_full_train + 1) // 2).astype(np.int8)
y_test_np01 = ((y_test_np + 1) // 2).astype(np.int8)

X_train_np, X_val_np, y_train_np, y_val_np = train_test_split(
    X_full_train,
    y_full_train,
    test_size=0.2,
    random_state=42,
    stratify=y_full_train
)

print(f'Forma de X_full_train: {X_full_train.shape}')
print(f'Forma de X_test: {X_test_np.shape}')
print('Distribucion de clases en train:')
print(y_train.value_counts())
print('Distribucion de clases en test:')
print(y_test.value_counts())
print('\nNota metodologica: igual que en LSTM, la CNN-1D trabaja con ventanas deslizantes sobre filas consecutivas del dataset reducido. Conviene presentarla como comparativa exploratoria de deep learning, no como modelo secuencial plenamente temporal.')


In [ ]:
# ==========================================
# 2. OPTUNA MULTIOBJETIVO: F1 Y LATENCIA
# ==========================================

def objective(trial):
    tf.keras.backend.clear_session()

    time_steps = trial.suggest_int('time_steps', 5, 15)
    n_filters = trial.suggest_int('n_filters', 32, 128, step=32)
    kernel_size = trial.suggest_int('kernel_size', 2, min(5, time_steps))
    dense_units = trial.suggest_int('dense_units', 16, 96, step=16)
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    f1_scores = []
    inference_times = []

    for train_idx, val_idx in skf.split(X_full_train, y_full_train_01):
        X_train_fold = X_full_train[train_idx]
        y_train_fold = y_full_train_01[train_idx]
        X_val_fold = X_full_train[val_idx]
        y_val_fold = y_full_train_01[val_idx]

        scaler = StandardScaler()
        X_train_fold_scaled = scaler.fit_transform(X_train_fold)
        X_val_fold_scaled = scaler.transform(X_val_fold)

        X_train_seq, y_train_seq = create_sequences(X_train_fold_scaled, y_train_fold, time_steps)
        X_val_seq, y_val_seq = create_sequences(X_val_fold_scaled, y_val_fold, time_steps)

        with tf.device(TRAIN_DEVICE):
            model = build_cnn1d_model(
                time_steps=time_steps,
                n_features=X_train_seq.shape[2],
                n_filters=n_filters,
                kernel_size=kernel_size,
                dense_units=dense_units,
                dropout_rate=DEFAULT_DROPOUT_RATE
            )

            early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
            model.fit(
                X_train_seq,
                y_train_seq,
                validation_split=0.1,
                epochs=20,
                batch_size=1024,
                callbacks=[early_stop],
                verbose=0
            )

        cpu_model = clone_model_to_cpu(
            trained_model=model,
            time_steps=time_steps,
            n_features=X_train_seq.shape[2],
            n_filters=n_filters,
            kernel_size=kernel_size,
            dense_units=dense_units,
            dropout_rate=DEFAULT_DROPOUT_RATE
        )

        with tf.device(INFER_DEVICE):
            t0 = time.perf_counter()
            y_pred_prob = cpu_model.predict(X_val_seq, batch_size=1024, verbose=0).ravel()
            t1 = time.perf_counter()

        y_pred = (y_pred_prob > 0.5).astype(np.int8)
        f1_scores.append(f1_score(y_val_seq, y_pred, zero_division=0))
        inference_times.append((t1 - t0) / len(X_val_seq))

        tf.keras.backend.clear_session()

    return float(np.mean(f1_scores)), float(np.mean(inference_times))


study = optuna.create_study(directions=['maximize', 'minimize'], study_name='cnn1d_ids_optimization')
print('Iniciando barrido multiobjetivo con CNN-1D (entrenamiento en GPU si existe, inferencia medida en CPU)...')
study.optimize(objective, n_trials=25, show_progress_bar=True)

results = []
pareto_trials = {trial.number for trial in study.best_trials}
for trial in study.trials:
    if trial.values is None:
        continue

    row = {
        'Trial': trial.number,
        'F1_CV': float(trial.values[0]),
        'Latencia_ms': float(trial.values[1] * 1000),
        'Pareto': trial.number in pareto_trials
    }
    row.update(trial.params)
    results.append(row)

df_cnn_trials = pl.DataFrame(results).sort(['Pareto', 'F1_CV', 'Latencia_ms'], descending=[True, True, False])
df_cnn_trials.write_csv('cnn1d_trials_results_cv.csv')
print("\nResultados guardados en 'cnn1d_trials_results_cv.csv'")
print(df_cnn_trials)


In [ ]:
# ==========================================
# 3. FRONTERA DE PARETO Y SELECCION DE CANDIDATOS
# ==========================================

df_pareto = df_cnn_trials.filter(pl.col('Pareto') == True).sort('Latencia_ms')
print(df_pareto)

plt.figure(figsize=(9, 6))
plt.scatter(df_cnn_trials['Latencia_ms'], df_cnn_trials['F1_CV'], c='lightgray', label='Trials')
plt.scatter(df_pareto['Latencia_ms'], df_pareto['F1_CV'], c='red', label='Frontera de Pareto')
plt.xlabel('Latencia de inferencia (ms por muestra)')
plt.ylabel('F1 medio en CV')
plt.title('CNN-1D: F1 vs Latencia')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

pareto_rows = df_pareto.to_dicts()

fastest_cfg = min(pareto_rows, key=lambda x: x['Latencia_ms'])
best_f1_cfg = max(pareto_rows, key=lambda x: x['F1_CV'])

f1_vals = np.array([row['F1_CV'] for row in pareto_rows], dtype=float)
lat_vals = np.array([row['Latencia_ms'] for row in pareto_rows], dtype=float)
f1_norm = (f1_vals - f1_vals.min()) / (f1_vals.max() - f1_vals.min() + 1e-12)
lat_norm = (lat_vals - lat_vals.min()) / (lat_vals.max() - lat_vals.min() + 1e-12)
balanced_idx = int(np.argmax(f1_norm - lat_norm))
balanced_cfg = pareto_rows[balanced_idx]

candidate_configs = {
    'Rapido': fastest_cfg,
    'Balanceado': balanced_cfg,
    'Mejor F1': best_f1_cfg
}

print('Candidatos seleccionados automaticamente desde la frontera de Pareto:')
for nombre, cfg in candidate_configs.items():
    print(nombre, cfg)


In [ ]:
# ==========================================
# 4. EVALUACION FINAL DE LOS CANDIDATOS EN TEST
# ==========================================

candidate_results = []

for perfil, cfg in candidate_configs.items():
    tf.keras.backend.clear_session()

    time_steps = int(cfg['time_steps'])
    n_filters = int(cfg['n_filters'])
    kernel_size = int(cfg['kernel_size'])
    dense_units = int(cfg['dense_units'])
    scaler = StandardScaler()
    X_full_train_scaled = scaler.fit_transform(X_full_train)
    X_test_scaled = scaler.transform(X_test_np)

    X_train_seq, y_train_seq = create_sequences(X_full_train_scaled, y_full_train_01, time_steps)
    X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_np01, time_steps)

    with tf.device(TRAIN_DEVICE):
        model = build_cnn1d_model(
            time_steps=time_steps,
            n_features=X_train_seq.shape[2],
            n_filters=n_filters,
            kernel_size=kernel_size,
            dense_units=dense_units,
            dropout_rate=DEFAULT_DROPOUT_RATE
        )

        early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
        model.fit(
            X_train_seq,
            y_train_seq,
            validation_split=0.1,
            epochs=20,
            batch_size=1024,
            callbacks=[early_stop],
            verbose=0
        )

    cpu_model = clone_model_to_cpu(
        trained_model=model,
        time_steps=time_steps,
        n_features=X_train_seq.shape[2],
        n_filters=n_filters,
        kernel_size=kernel_size,
        dense_units=dense_units,
        dropout_rate=DEFAULT_DROPOUT_RATE
    )

    with tf.device(INFER_DEVICE):
        t0 = time.perf_counter()
        y_prob = cpu_model.predict(X_test_seq, batch_size=1024, verbose=0).ravel()
        t1 = time.perf_counter()
    y_pred = (y_prob > 0.5).astype(np.int8)

    candidate_results.append({
        'Perfil': perfil,
        'Trial': int(cfg['Trial']),
        'time_steps': time_steps,
        'n_filters': n_filters,
        'kernel_size': kernel_size,
        'dense_units': dense_units,
        'F1_Test': float(f1_score(y_test_seq, y_pred, zero_division=0)),
        'Accuracy_Test': float(accuracy_score(y_test_seq, y_pred)),
        'Latencia_ms': float(((t1 - t0) / len(X_test_seq)) * 1000)
    })

df_candidate_results = pl.DataFrame(candidate_results).sort(['F1_Test', 'Latencia_ms'], descending=[True, False])
print(df_candidate_results)

# Puedes cambiar manualmente esta eleccion si quieres usar otro perfil de la frontera.
best_c = candidate_configs['Mejor F1']
print('\nModelo ganador seleccionado para ROC, matriz de confusion y benchmark:')
print(best_c)


In [ ]:
# ==========================================
# 5. ROC, MATRIZ DE CONFUSION Y BENCHMARK FINAL
# ==========================================

tf.keras.backend.clear_session()

time_steps = int(best_c['time_steps'])
n_filters = int(best_c['n_filters'])
kernel_size = int(best_c['kernel_size'])
dense_units = int(best_c['dense_units'])
scaler_best = StandardScaler()
X_full_train_scaled = scaler_best.fit_transform(X_full_train)
X_test_scaled = scaler_best.transform(X_test_np)

X_train_seq, y_train_seq = create_sequences(X_full_train_scaled, y_full_train_01, time_steps)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_np01, time_steps)

with tf.device(TRAIN_DEVICE):
    model_final = build_cnn1d_model(
        time_steps=time_steps,
        n_features=X_train_seq.shape[2],
        n_filters=n_filters,
        kernel_size=kernel_size,
        dense_units=dense_units,
        dropout_rate=DEFAULT_DROPOUT_RATE
    )

    early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    model_final.fit(
        X_train_seq,
        y_train_seq,
        validation_split=0.1,
        epochs=20,
        batch_size=1024,
        callbacks=[early_stop],
        verbose=0
    )

model_final_cpu = clone_model_to_cpu(
    trained_model=model_final,
    time_steps=time_steps,
    n_features=X_train_seq.shape[2],
    n_filters=n_filters,
    kernel_size=kernel_size,
    dense_units=dense_units,
    dropout_rate=DEFAULT_DROPOUT_RATE
)

# ==========================================
# 5.1 BENCHMARK DE RECURSOS COMPUTACIONALES
# ==========================================
print('Midiendo recursos computacionales del modelo ganador (CNN-1D)...')
proceso = psutil.Process(os.getpid())
block_size = 2048
repetitions = 3

with tf.device(INFER_DEVICE):
    _ = model_final_cpu.predict(X_test_seq[:min(512, len(X_test_seq))], batch_size=512, verbose=0)

tiempos_muro = []
tiempos_cpu = []
picos_ram = []

for _ in range(repetitions):
    cpu_ini = proceso.cpu_times()
    ram_base = proceso.memory_info().rss / (1024 * 1024)
    pico_ram_rep = ram_base

    t0 = time.perf_counter()
    for inicio in range(0, len(X_test_seq), block_size):
        fin = inicio + block_size
        bloque = X_test_seq[inicio:fin]
        with tf.device(INFER_DEVICE):
            _ = model_final_cpu.predict(bloque, batch_size=512, verbose=0)

        ram_actual = proceso.memory_info().rss / (1024 * 1024)
        if ram_actual > pico_ram_rep:
            pico_ram_rep = ram_actual

    t1 = time.perf_counter()
    cpu_fin = proceso.cpu_times()

    tiempos_muro.append(t1 - t0)
    tiempos_cpu.append((cpu_fin.user - cpu_ini.user) + (cpu_fin.system - cpu_ini.system))
    picos_ram.append(pico_ram_rep - ram_base)

media_muro = float(np.mean(tiempos_muro))
media_cpu = float(np.mean(tiempos_cpu))
pico_max_ram = float(np.max(picos_ram))
total_nucleos = psutil.cpu_count(logical=True)

df_benchmark_cnn1d = pl.DataFrame([{
    'Modelo': 'CNN-1D',
    'Latencia_ms': round((media_muro / len(X_test_seq)) * 1000, 5),
    'Thruput (paq/s)': round(len(X_test_seq) / media_muro, 0),
    'Nucleos CPU': round(media_cpu / media_muro if media_muro > 0 else 1, 1),
    'Pico RAM (MB)': round(pico_max_ram, 2),
    'Porcentaje CPU': round(((media_cpu / media_muro) / total_nucleos) * 100 if media_muro > 0 else 0, 1)
}])
print(df_benchmark_cnn1d)

# ==========================================
# 5.2 PREDICCIONES Y GRAFICAS
# ==========================================
with tf.device(INFER_DEVICE):
    y_pred_prob = model_final_cpu.predict(X_test_seq, batch_size=1024, verbose=0).ravel()
y_pred = (y_pred_prob > 0.5).astype(np.int8)

cm = confusion_matrix(y_test_seq, y_pred)
fpr, tpr, _ = roc_curve(y_test_seq, y_pred_prob)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    ax=ax[0],
    xticklabels=['Ataque (0)', 'Normal (1)'],
    yticklabels=['Ataque (0)', 'Normal (1)']
)
ax[0].set_title('Matriz de Confusion - CNN-1D')
ax[0].set_xlabel('Prediccion')
ax[0].set_ylabel('Realidad')

ax[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'Curva ROC (AUC = {roc_auc:.4f})')
ax[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Clasificador aleatorio')
ax[1].set_xlim([0.0, 1.0])
ax[1].set_ylim([0.0, 1.05])
ax[1].set_xlabel('Tasa de Falsos Positivos (FPR)')
ax[1].set_ylabel('Tasa de Verdaderos Positivos (TPR)')
ax[1].set_title('Curva ROC - CNN-1D')
ax[1].legend(loc='lower right')
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
